# ByT5 Turkish Diacritics Restoration

**Goal:** Fine-tune `google/byt5-small` to restore Turkish diacritics from ASCII text.

**Hardware:** Kaggle T4 GPU
**Dataset:** 2,942-entry ambiguous-lookup.tsv

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import torch
import glob
import random
from transformers import T5ForConditionalGeneration, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Model

In [ ]:
MODEL_NAME = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Load Dataset

In [ ]:
# Load sentence-level training data (from Turkish content + words + negatives)
TRAINING_PATH = glob.glob("/kaggle/input/**/training_data.tsv", recursive=True)
if TRAINING_PATH:
    TRAINING_PATH = TRAINING_PATH[0]
    print(f"Training data: {TRAINING_PATH}")
else:
    TRAINING_PATH = "/kaggle/input/datasets/ceaksan/training-data-from-content/training_data.tsv"
    print(f"Using default: {TRAINING_PATH}")

def load_tsv(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pairs.append((parts[0], [parts[1]]))
    return pairs

pairs = load_tsv(TRAINING_PATH)
print(f"Loaded {len(pairs)} pairs")
print(f"Examples:")
for src, tgt in pairs[:3]:
    print(f"  {src[:60]}... -> {tgt[0][:60]}...")

## 3. Prepare Dataset

In [ ]:
class DiacriticsDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=128, prefix="restore: "):
        self.data = []
        for ascii_form, correct_forms in pairs:
            for correct in correct_forms:
                self.data.append((prefix + ascii_form, correct))
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text, tgt_text = self.data[idx]
        src = self.tokenizer(src_text, max_length=self.max_len, padding="max_length",
                             truncation=True, return_tensors="pt")
        tgt = self.tokenizer(tgt_text, max_length=self.max_len, padding="max_length",
                             truncation=True, return_tensors="pt")
        labels = tgt["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": src["input_ids"].squeeze(),
            "attention_mask": src["attention_mask"].squeeze(),
            "labels": labels
        }

random.seed(42)
shuffled = pairs[:]
random.shuffle(shuffled)
split = int(len(shuffled) * 0.9)
train_pairs = shuffled[:split]
eval_pairs = shuffled[split:]

train_dataset = DiacriticsDataset(train_pairs, tokenizer)
eval_dataset = DiacriticsDataset(eval_pairs, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=8)

print(f"Train: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Eval:  {len(eval_dataset)} samples ({len(eval_loader)} batches)")

## 4. Helpers

In [ ]:
@torch.no_grad()
def predict(text, model, tokenizer, device=DEVICE, prefix="restore: "):
    inputs = tokenizer(prefix + text, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=len(text) + 10, num_beams=1, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def benchmark(pairs, model, tokenizer, device=DEVICE):
    correct = partial = 0
    total = len(pairs)
    errors = []
    for ascii_form, expected_forms in pairs:
        try:
            result = predict(ascii_form, model, tokenizer, device).strip()
            if result in expected_forms:
                correct += 1
            elif any(ef in result or result in ef for ef in expected_forms):
                partial += 1
                errors.append((ascii_form, expected_forms, result, "partial"))
            else:
                errors.append((ascii_form, expected_forms, result, "wrong"))
        except Exception as e:
            errors.append((ascii_form, expected_forms, str(e), "error"))
    print(f"\nBenchmark ({total} pairs)")
    print(f"  Exact:   {correct}/{total} ({correct/total*100:.1f}%)")
    print(f"  Partial: {partial}/{total} ({partial/total*100:.1f}%)")
    print(f"  Wrong:   {len(errors)-partial}/{total} ({(len(errors)-partial)/total*100:.1f}%)")
    for a, e, g, t in errors[:10]:
        print(f"  [{t}] {a} -> {g} (expected: {e})")
    return correct, partial, errors

## 5. Fine-Tune (fp32, no AMP)

In [ ]:
def fine_tune(model, train_loader, eval_loader, epochs=10, lr=3e-4, device=DEVICE):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    best_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_train = total_loss / len(train_loader)

        model.eval()
        eval_loss = 0
        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                eval_loss += outputs.loss.item()
        avg_eval = eval_loss / len(eval_loader)

        marker = " *" if avg_eval < best_loss else ""
        if avg_eval < best_loss:
            best_loss = avg_eval
        print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train:.4f} | Eval: {avg_eval:.4f}{marker}")

    return model

model = fine_tune(model, train_loader, eval_loader, epochs=10, lr=3e-4)

## 6. Results

In [ ]:
model.eval()

print("Post fine-tune test:")
for w in ["turkce", "acar", "gunes", "ogrenci", "calisma", "urun", "gorusuyoruz"]:
    print(f"  {w} -> {predict(w, model, tokenizer)}")

print("\nFalse positive test:")
for w in ["Opus", "log", "Python", "API", "cache"]:
    r = predict(w, model, tokenizer)
    print(f"  {w} -> {r} [{'OK' if r == w else 'WRONG'}]")

print("\nEval set:")
benchmark(eval_pairs, model, tokenizer)

print("\nFull dataset:")
benchmark(pairs, model, tokenizer)

## 7. Save

In [ ]:
OUTPUT_DIR = "/kaggle/working/byt5-turkish-diacritics"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")